This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log, pi
import shutil
from pathlib import Path
from enum import Enum

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences

# from data_processing.paths import (
#     get_report_root, get_exp_root, get_reactor_data_root, NEUTR)
from data_processing.dot_env import config
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, weight_factor, stopping_criteria, _nan_divide, cut_low_l, UnfoldingProcessInfo

In [ ]:
bins_min = helpers.get_input_with_default(
    "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
    0,
    float
)
bins_max = helpers.get_input_with_default(
    "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
    1.2,
    float
)
bins_width = helpers.get_input_with_default(
    "Enter light output bin width (in MeVee), or press Enter for default (0.01 MeVee)",
    0.01,
    float
)

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

In [ ]:
main_data_folder = Path(config["NEUTRON_DATA_FOLDER"])

R = load_neutron_response_matrix(
    main_data_folder / "response_matrix_R4_mono",
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
test_file = main_data_folder / "sigma_50keV_FWHM.csv.npy"

L_array = np.load(test_file)

bins = np.arange(bins_min, bins_max + bins_width, bins_width)
np_cps, *_ = np.histogram(L_array, bins=bins)

# For SI Fig 7, adjust L_array so total sim counts approx. = avg. experimental counts
cps_sum = np_cps.sum()
exp_sum = 1038527.6
cps_corr_factor = exp_sum / cps_sum
print(cps_corr_factor)
np_cps = np_cps * cps_corr_factor
print(np_cps.sum())

np_cps = np_cps.reshape(-1, 1)
np_Ls = (bins[1:] + bins[:-1]) / 2

ddN = NDHistogram(np_cps, [np_Ls, np.ones(1)])

In [ ]:
dd_phi, _ = unfold_spectrum(
    R,
    ddN,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
)

In [ ]:
dd_phi_flat = dd_phi.counts.reshape(-1)
dd_phi_mids = dd_phi.midpoints[1]

In [ ]:
base_xerr = [(1.798, (0.31000000000000005, 0.6200000000000001)),
 (1.86, (0.31000000000000005, 0.558)),
 (1.922, (0.3719999999999999, 0.558)),
 (1.984, (0.3719999999999999, 0.496)),
 (2.046, (0.3719999999999999, 0.496)),
 (2.108, (0.3720000000000001, 0.496)),
 (2.17, (0.43399999999999994, 0.43400000000000016)),
 (2.232, (0.3720000000000001, 0.3719999999999999)),
 (2.294, (0.3720000000000001, 0.3719999999999999)),
 (2.418, (0.43400000000000016, 0.31000000000000005)),
 (2.48, (0.496, 0.31000000000000005)),
 (2.604, (0.3719999999999999, 0.4339999999999997)),
 (2.666, (0.4339999999999997, 0.3719999999999999)),
 (2.79, (0.496, 0.3719999999999999)),
 (2.852, (0.3719999999999999, 0.43400000000000016)),
 (2.914, (0.43400000000000016, 0.3719999999999999)),
 (2.976, (0.496, 0.18599999999999994)),
 (3.038, (0.496, 0.1860000000000004)),
 (3.1, (0.5580000000000003, 0.18599999999999994)),
 (3.162, (0.496, 0.18599999999999994)),
 (3.224, (0.5580000000000003, 0.18599999999999994)),
 (3.286, (0.5579999999999998, 0.18599999999999994))]

In [ ]:
dd_xerr = []
for mid in dd_phi_mids:
    isclose = [np.isclose(x, mid) for x, _ in base_xerr]
    true_idx = [i for i, x in enumerate(isclose) if x]
    if len(true_idx) == 0:
        dd_xerr.append((np.nan, np.nan))
        continue
    idx = true_idx[0]
    _, errorbar = base_xerr[idx]
    dd_xerr.append(errorbar)
dd_xerr = list(zip(*dd_xerr))

In [ ]:
dd_sigma_counts = np.sqrt(ddN.counts)
dd_sigma = NDHistogram(dd_sigma_counts, ddN.midpoints)
ddN_yerr_plus = NDHistogram(ddN.counts + 3 * dd_sigma_counts, ddN.midpoints)
ddN_yerr_minus = NDHistogram(ddN.counts - 3 * dd_sigma_counts, ddN.midpoints)

ddphi_yerr_plus, _ = unfold_spectrum(
    R,
    ddN_yerr_plus,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
    sigma=dd_sigma
)
ddphi_yerr_minus, _ = unfold_spectrum(
    R,
    ddN_yerr_minus,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
    sigma=dd_sigma
)
ddphi_yerr_plus_flat = ddphi_yerr_plus.counts.reshape(-1)
ddphi_yerr_minus_flat = ddphi_yerr_minus.counts.reshape(-1)

plus_is_larger = ddphi_yerr_plus_flat >= ddphi_yerr_minus_flat
plus_gt_phi = (ddphi_yerr_plus_flat >= dd_phi_flat) | (np.isnan(dd_phi_flat))
minus_lt_phi = (ddphi_yerr_minus_flat <= dd_phi_flat) | (np.isnan(dd_phi_flat))
ddphi_yerr_larger = ddphi_yerr_plus_flat.copy()
ddphi_yerr_larger[~plus_gt_phi] = ddphi_yerr_minus_flat[~plus_gt_phi]
ddphi_yerr_smaller = ddphi_yerr_minus_flat.copy()
ddphi_yerr_smaller[~minus_lt_phi] = ddphi_yerr_plus_flat[~minus_lt_phi]

dd_yerr_up = np.nan_to_num(ddphi_yerr_larger - dd_phi_flat, nan=np.nan)
dd_yerr_down = np.nan_to_num(dd_phi_flat - ddphi_yerr_smaller, nan=np.nan)
dd_yerr_up[dd_yerr_up < 0] = 0
dd_yerr_down[dd_yerr_down < 0] = 0

dd_xerr_left, _ = dd_xerr
dd_yerr_down[np.isnan(dd_xerr_left)] = np.nan
dd_yerr_up[np.isnan(dd_xerr_left)] = np.nan

dd_yerr = [dd_yerr_down, dd_yerr_up]

In [ ]:
#linalg normalization test
# mono_norm = np.linalg.norm(mono_phi_flat[~np.isnan(mono_phi_flat)])
# mono_phi_normed = mono_phi_flat/mono_norm

dd_norm = np.linalg.norm(dd_phi_flat[~np.isnan(dd_phi_flat)])
dd_phi_normed = dd_phi_flat / dd_norm
dd_N_counts = np.nansum(ddN.counts)
dd_phi_norm_sum = np.nansum(dd_phi_normed)
count_adj_factor = dd_N_counts / dd_phi_norm_sum
dd_phi_normed = dd_phi_normed * count_adj_factor

d_detector = 12.7  # cm
cs_area = pi * d_detector * d_detector / 4
dd_phi_fluence = dd_phi_normed / cs_area

dd_yerr = [
    ((err / dd_norm) * count_adj_factor) / cs_area
    for err in dd_yerr
]

In [ ]:
figsize=(9,6)
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)
ax.errorbar(
    dd_phi_mids,
    # dd_phi_normed,
    dd_phi_fluence,
    xerr=dd_xerr,
    yerr=dd_yerr,
    marker="o", markersize=3, ecolor="black",
    markerfacecolor="red",
    markeredgecolor="red",
    label="DD fusion simulation"
)

ax.set_xlabel("Neutron energy (MeV)", fontsize=fontsize)
ax.set_ylabel(r"Neutron fluence (1/cm${^2}$)", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)
plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()